In [1]:
import time
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, explode, split, broadcast
from pyspark.sql.window import Window

In [ ]:
#Fetch IMDb Dataset using wget
#!wget -q https://datasets.imdbws.com/title.basics.tsv.gz
!wget -q https://datasets.imdbws.com/title.ratings.tsv.gz

In [ ]:
#Initialize Spark Session
# Create a Spark session for processing the data
spark = SparkSession.builder.master("local[16]").appName("IMDB_Optimization").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/04 19:56:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
#Load IMDb Data
# Read the IMDb data as DataFrames
start_time = time.time()
titles_df = spark.read.option("sep", "\t").option("header", "true").option("nullValue", r"\N").csv("title.basics.tsv.gz")
ratings_df = spark.read.option("sep", "\t").option("header", "true").option("nullValue", r"\N").csv("title.ratings.tsv.gz")
titles_df = titles_df.select("tconst", "primaryTitle", "startYear", "titleType", "genres")
ratings_df = ratings_df.select("tconst", "averageRating", "numVotes")
load_time = time.time() - start_time

In [5]:
# Naive Query Execution (No Optimization)
# Join the datasets without filtering early (inefficient method)
start_time = time.time()
all_movies = titles_df.join(ratings_df, "tconst")
filtered = all_movies.filter((col("startYear") >= 2010) & (col("numVotes") >= 10000))
filtered = filtered.withColumn("genre", explode(split(col("genres"), ",")))
window = Window.partitionBy("genre").orderBy(col("averageRating").desc(), col("numVotes").desc())
top10_naive = filtered.withColumn("rank", count("primaryTitle").over(window)).filter(col("rank") <= 10)
top10_naive.count()
naive_time = time.time() - start_time

In [6]:
#Optimization 1 - Early Filtering
# Reduce data size before the join to minimize processing time
start_time = time.time()
titles_filtered = titles_df.filter((col("startYear") >= 2010) & (col("titleType") == "movie"))
ratings_filtered = ratings_df.filter(col("numVotes") >= 10000)
opt1_time = time.time() - start_time

In [7]:
#Optimization 2 - Broadcast Join
# Use broadcast join to avoid costly shuffles
start_time = time.time()
optimized_df = titles_filtered.join(broadcast(ratings_filtered), "tconst")
opt2_time = time.time() - start_time

In [9]:
#Compute Optimized Top 10 per Genre
# Apply genre explosion and ranking on the optimized dataset
start_time = time.time()
optimized_filtered = optimized_df.withColumn("genre", explode(split(col("genres"), ",")))
top10_optimized = optimized_filtered.withColumn("rank", count("primaryTitle").over(window)).filter(col("rank") <= 10)
top10_optimized.count()
opt4_time = time.time() - start_time

In [8]:
#Optimization 3 - Partitioning and Pushdown
# Store data in a partitioned format for faster access
start_time = time.time()
titles_filtered.write.mode("overwrite").partitionBy("startYear").parquet("titles_partitioned.parquet")
titles_partitioned = spark.read.parquet("titles_partitioned.parquet")
titles_partitioned.createOrReplaceTempView("titles")
titles_partitioned = spark.sql("SELECT * FROM titles WHERE startYear >= 2010")
opt3_time = time.time() - start_time

In [10]:
#Generate Comparison Table
# Collect execution time metrics to analyze optimization impact
results = pd.DataFrame({
    "Optimization Step": ["Naive Query", "Early Filtering", "Broadcast Join", "Partitioning", "Final Optimized Query"],
    "Execution Time (s)": [naive_time, opt1_time, opt2_time, opt3_time, opt4_time]
})

In [11]:
# Additional Metrics
# Measure dataset sizes after each optimization to analyze efficiency
titles_size = titles_df.count()
ratings_size = ratings_df.count()
filtered_titles_size = titles_filtered.count()
filtered_ratings_size = ratings_filtered.count()
optimized_size = optimized_df.count()

In [14]:
results["Data Processed (Rows)"] = [titles_size, filtered_titles_size, filtered_ratings_size, optimized_size, optimized_size]

In [12]:
# Display Results in DataFrame Format
print("\nOptimization Impact Comparison:\n")
print(results.to_string(index=False))

# Save Results as CSV
results.to_csv("spark_optimization_comparison.csv", index=False)


Optimization Impact Comparison:

    Optimization Step  Execution Time (s)
          Naive Query          111.001424
      Early Filtering            0.042083
       Broadcast Join            0.046065
         Partitioning           82.648450
Final Optimized Query           82.059726


In [13]:
spark.stop()